In [ ]:
import pandas as pd

# 示例数据
customers = pd.DataFrame({
    'customer_id': [1, 2, 3, 4, 5],
    'name': ['Winston', 'Jonathan', 'Annabelle', 'Marwan', 'Khaled']
})

orders = pd.DataFrame({
    'order_id': [1,2,3,4,5,6,7,8,9,10],
    'order_date': pd.to_datetime(['2020-07-31','2020-07-30','2020-08-29','2020-07-29',
                                  '2020-06-10','2020-08-01','2020-08-01','2020-08-03',
                                  '2020-08-07','2020-07-15']),
    'customer_id': [1,2,3,4,1,2,3,1,2,1],
    'product_id': [1,2,3,1,2,1,1,2,3,2]
})

products = pd.DataFrame({
    'product_id': [1,2,3,4],
    'product_name': ['keyboard','mouse','screen','hard disk'],
    'price': [120,80,600,450]
})

# 1. 合并 Orders 和 Products 获取 product_name
df = orders.merge(products[['product_id','product_name']], on='product_id', how='left')

# 2. 找到每个 product_id 的最新 order_date
latest_dates = df.groupby('product_id')['order_date'].max().reset_index()
latest_dates.rename(columns={'order_date':'latest_order_date'}, inplace=True)

# 3. 合并回 df，筛选最新订单
df = df.merge(latest_dates, on='product_id')
latest_orders = df[df['order_date'] == df['latest_order_date']]

# 4. 选择需要列并排序
result = latest_orders[['product_name','product_id','order_id','order_date']].sort_values(
    by=['product_name','product_id','order_id']
).reset_index(drop=True)

print(result)


In [ ]:
import pandas as pd

# Users 表
users = pd.DataFrame({
    'user_id': [1, 2, 3, 4],
    'user_name': ['Moustafa', 'Jonathan', 'Winston', 'Luis'],
    'credit': [100, 200, 10000, 800]
})

# Transactions 表
transactions = pd.DataFrame({
    'trans_id': [1, 2, 3],
    'paid_by': [1, 3, 2],
    'paid_to': [3, 2, 1],
    'amount': [400, 500, 200],
    'transacted_on': ['2020-08-01', '2020-08-02', '2020-08-03']
})

# 计算每个用户支出总额
spent = transactions.groupby('paid_by')['amount'].sum().reset_index()
spent.rename(columns={'amount': 'spent'}, inplace=True)

# 计算每个用户收入总额
received = transactions.groupby('paid_to')['amount'].sum().reset_index()
received.rename(columns={'amount': 'received'}, inplace=True)

# 合并 users、spent、received
df = users.merge(spent, left_on='user_id', right_on='paid_by', how='left') \
          .merge(received, left_on='user_id', right_on='paid_to', how='left')

# 支出或收入为空时设为 0
df['spent'] = df['spent'].fillna(0)
df['received'] = df['received'].fillna(0)

# 计算交易后的余额
df['credit'] = df['credit'] - df['spent'] + df['received']

# 判断是否透支
df['credit_limit_breached'] = df['credit'].apply(lambda x: 'Yes' if x < 0 else 'No')

# 选取最终列
result = df[['user_id', 'user_name', 'credit', 'credit_limit_breached']]

print(result)


In [ ]:
import pandas as pd

# 示例数据
customers = pd.DataFrame({
    'customer_id': [1, 2, 3, 4, 5],
    'name': ['Alice', 'Bob', 'Tom', 'Jerry', 'John']
})

orders = pd.DataFrame({
    'order_id': [1,2,3,4,5,6,7,8,9,10],
    'order_date': ['2020-07-31','2020-07-30','2020-08-29','2020-07-29','2020-06-10',
                   '2020-08-01','2020-08-01','2020-08-03','2020-08-07','2020-07-15'],
    'customer_id': [1,2,3,4,1,2,3,1,2,1],
    'product_id': [1,2,3,1,2,1,3,2,3,2]
})

products = pd.DataFrame({
    'product_id': [1,2,3,4],
    'product_name': ['keyboard','mouse','screen','hard disk'],
    'price': [120,80,600,450]
})

# 1. 统计每个顾客每个商品的下单次数
order_counts = orders.groupby(['customer_id','product_id']).size().reset_index(name='count')

# 2. 找出每个顾客的最大下单次数
max_counts = order_counts.groupby('customer_id')['count'].transform('max')
most_frequent = order_counts[order_counts['count'] == max_counts]

# 3. 与Products表连接，获取商品名称
result = most_frequent.merge(products[['product_id','product_name']], on='product_id', how='left')

# 4. 选择所需列
result = result[['customer_id','product_id','product_name']]

print(result)
